# Final Yahoo Finance Data Download

This notebook downloads daily financial data for:
- Bitcoin (`BTC-USD`)
- Gold Futures (`GC=F`)
- S&P 500 Index (`^GSPC`)

The period starts before the first Polymarket observation so lagged variables can be calculated later.

In [ ]:
# ============================================================
# STEP 1: Install and Import Required Libraries
# ============================================================

!pip install -q yfinance openpyxl

import yfinance as yf
import pandas as pd
import numpy as np


In [ ]:
# ============================================================
# STEP 2: Define Financial Assets and Date Range
# ============================================================

START_DATE = "2025-06-01"

# yfinance end date is exclusive.
# Therefore 2026-08-12 includes observations through 2026-08-11.
END_DATE = "2026-08-12"

tickers = {
    "Bitcoin": "BTC-USD",
    "Gold": "GC=F",
    "SP500": "^GSPC"
}

print("Assets:")
for asset, ticker in tickers.items():
    print(f"{asset}: {ticker}")


In [ ]:
# ============================================================
# STEP 3: Download Bitcoin Daily Data
# ============================================================

bitcoin = yf.download(
    "BTC-USD",
    start=START_DATE,
    end=END_DATE,
    interval="1d",
    auto_adjust=False,
    repair=True,
    progress=False
)

if isinstance(bitcoin.columns, pd.MultiIndex):
    bitcoin.columns = bitcoin.columns.get_level_values(0)

bitcoin = bitcoin.reset_index()
bitcoin.head()


In [ ]:
# ============================================================
# STEP 4: Download Gold Daily Data
# ============================================================

gold = yf.download(
    "GC=F",
    start=START_DATE,
    end=END_DATE,
    interval="1d",
    auto_adjust=False,
    repair=True,
    progress=False
)

if isinstance(gold.columns, pd.MultiIndex):
    gold.columns = gold.columns.get_level_values(0)

gold = gold.reset_index()
gold.head()


In [ ]:
# ============================================================
# STEP 5: Download S&P 500 Daily Data
# ============================================================

sp500 = yf.download(
    "^GSPC",
    start=START_DATE,
    end=END_DATE,
    interval="1d",
    auto_adjust=False,
    repair=True,
    progress=False
)

if isinstance(sp500.columns, pd.MultiIndex):
    sp500.columns = sp500.columns.get_level_values(0)

sp500 = sp500.reset_index()
sp500.head()


In [ ]:
# ============================================================
# STEP 6: Inspect Downloaded Financial Data
# ============================================================

datasets = {
    "Bitcoin": bitcoin,
    "Gold": gold,
    "S&P 500": sp500
}

for name, df in datasets.items():
    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)
    print("Rows:", len(df))
    print("First date:", df["Date"].min())
    print("Last date:", df["Date"].max())
    print("\nMissing values:")
    print(df.isna().sum())
    print("\nFirst 3 rows:")
    display(df.head(3))


In [ ]:
# ============================================================
# STEP 7: Add Asset Names
# ============================================================

bitcoin["Asset"] = "Bitcoin"
gold["Asset"] = "Gold"
sp500["Asset"] = "SP500"

print("Bitcoin shape:", bitcoin.shape)
print("Gold shape:", gold.shape)
print("SP500 shape:", sp500.shape)


In [ ]:
# ============================================================
# STEP 8: Combine Financial Data into Long Format
# ============================================================

financial_raw = pd.concat(
    [bitcoin, gold, sp500],
    ignore_index=True
)

financial_raw = financial_raw.sort_values(
    ["Date", "Asset"]
).reset_index(drop=True)

financial_raw.head(10)


In [ ]:
# ============================================================
# STEP 9: Save Raw Yahoo Finance Dataset
# ============================================================

output_path = "/content/Final_Yahoo_Finance_BTC_Gold_SP500_Raw.xlsx"

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    financial_raw.to_excel(
        writer,
        sheet_name="All Financial Data",
        index=False
    )

    bitcoin.to_excel(
        writer,
        sheet_name="Bitcoin",
        index=False
    )

    gold.to_excel(
        writer,
        sheet_name="Gold",
        index=False
    )

    sp500.to_excel(
        writer,
        sheet_name="SP500",
        index=False
    )

print("Saved successfully:")
print(output_path)
